# Deep Learning PR 5: Sentiment Analysis with RNN, LSTM and GRU


## 0. Imports, reproducibility and configuration


In [3]:
import os, re, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models

warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = "IMDB Dataset.csv"
VOCAB_SIZE = 10000
EMBEDDING_DIM = 32
MAXLEN = 200       # Update after inspecting the 90th–95th percentile
BATCH_SIZE = 64
EPOCHS = 15

os.makedirs("plots", exist_ok=True)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Configuration:", VOCAB_SIZE, EMBEDDING_DIM, MAXLEN, BATCH_SIZE, EPOCHS)

TensorFlow: 2.21.0
NumPy: 2.4.6
Configuration: 10000 32 200 64 15


# Task 1 — Text loading, cleaning, tokenisation and padding
**Objective:** Prepare raw reviews without losing sentiment-bearing words or leaking test information into training.

## 1.1 Load and inspect
The dataset should contain `review` and `sentiment`. First inspect dimensions, examples, label balance and missing values.

In [7]:
# ==========================================
# TASK 1.1 - LOAD AND INSPECT IMDB DATASET
# ==========================================

import os
import glob
import kagglehub
import pandas as pd

# 1. Download dataset
path = kagglehub.dataset_download(
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
)

print("Dataset path:", path)

# 2. Find CSV file
csv_files = glob.glob(os.path.join(path, "*.csv"))

if not csv_files:
    raise FileNotFoundError("IMDB CSV file not found!")

# 3. Load dataset into DataFrame
df = pd.read_csv(csv_files[0])

# 4. Display dataset information
print("\nDataset Shape:", df.shape)

print("\nFirst 5 Reviews:")
display(df.head())

print("\nDataset Information:")
df.info()

# 5. Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# 6. Check sentiment distribution
print("\nSentiment Distribution:")
print(df["sentiment"].value_counts())

# 7. Remove duplicate reviews
rows_before = len(df)

duplicate_count = df.duplicated(
    subset=["review", "sentiment"]
).sum()

df = df.drop_duplicates(
    subset=["review", "sentiment"]
).reset_index(drop=True)

# 8. Display duplicate removal results
print("\n========== DUPLICATE REMOVAL ==========")
print("Rows Before:", rows_before)
print("Duplicate Rows:", duplicate_count)
print("Rows After:", len(df))

display(df.head())

Dataset path: C:\Users\lenovo\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1

Dataset Shape: (50000, 2)

First 5 Reviews:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 63.6 MB

Missing Values:
review       0
sentiment    0
dtype: int64

Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

========== DUPLICATE REMOVAL ==========
Rows Before: 50000
Duplicate Rows: 418
Rows After: 49582


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 1.3 Clean text
We lowercase text, remove HTML tags, retain letters and apostrophes, and collapse repeated spaces.

**Why retain apostrophes?** Contractions such as `don't` and `isn't` contain negation information.  
**Why not remove stop-words?** Words such as `not` and `no` can reverse sentiment, and word order matters for recurrent models.

In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"<.*?>", " ", s)           # remove HTML tags
    s = re.sub(r"[^a-zA-Z']", " ", s)      # keep letters and apostrophes
    s = re.sub(r"\\s+", " ", s).strip()  # collapse spaces
    return s

example_raw = df.loc[0, "review"]
print("BEFORE:\\n", example_raw[:700])
df["clean_review"] = df["review"].apply(clean_text)
print("\\nAFTER:\\n", df.loc[0, "clean_review"][:700])

df = df[df["clean_review"].str.len() > 0].reset_index(drop=True)
y = df["sentiment"].map({"positive": 1, "negative": 0}).astype("int32").to_numpy()
texts = df["clean_review"].astype(str).to_numpy()

## 1.4 Review-length analysis
Count words in each cleaned review. The histogram shows the distribution; the mean, median and 95th percentile help select a sequence length. A percentile-based length keeps most words while controlling memory and computation. Very long reviews will be truncated.

In [ ]:
df["word_count"] = df["clean_review"].str.split().str.len()
mean_len = df["word_count"].mean()
median_len = df["word_count"].median()
p95_len = df["word_count"].quantile(0.95)
print(f"Mean: {mean_len:.1f}, Median: {median_len:.1f}, 95th percentile: {p95_len:.1f}")

plt.figure(figsize=(10, 5))
sns.histplot(df["word_count"], bins=60, kde=True)
plt.axvline(mean_len, linestyle="--", label=f"Mean {mean_len:.1f}")
plt.axvline(median_len, linestyle="--", label=f"Median {median_len:.1f}")
plt.axvline(p95_len, linestyle="--", label=f"95th percentile {p95_len:.1f}")
plt.title("IMDB Review Length Distribution")
plt.xlabel("Words per review"); plt.ylabel("Number of reviews")
plt.legend(); plt.tight_layout()
plt.savefig("plots/length_distribution.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="sentiment", y="word_count")
plt.title("Review Length by Sentiment")
plt.xlabel("Sentiment"); plt.ylabel("Word count")
plt.tight_layout(); plt.show()

# Choose a rounded percentile-based length. Change this after examining the plots.
MAXLEN = int(np.ceil(p95_len / 10) * 10)
MAXLEN = max(50, min(MAXLEN, 500))
print("Selected MAXLEN:", MAXLEN)

## 1.5 Split first, then fit the tokenizer
The split must happen **before** tokenizer fitting. This prevents the vocabulary from learning word-frequency information from the test reviews. `num_words=10000` limits the vocabulary to frequent words; rare words add model parameters and may contribute noise. `<OOV>` represents words not retained in the vocabulary.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42, stratify=y
)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

train_sequences = tokenizer.texts_to_sequences(X_train_text)
test_sequences = tokenizer.texts_to_sequences(X_test_text)

print("Top 10 indexed words:", list(tokenizer.word_index.items())[:10])
print("Example integer sequence:", train_sequences[0][:50])

## 1.6 Pad and truncate
Each review must have the same length to form a batch tensor. `pre` padding adds zeros at the beginning; `pre` truncation keeps the final part of an overlong review. The assignment's rationale is that the final recurrent state should be based on real words rather than a long run of trailing zeros. In practice, masking can also be considered, but this notebook follows the specified configuration.

In [ ]:
X_train = pad_sequences(train_sequences, maxlen=MAXLEN, padding="pre", truncating="pre")
X_test = pad_sequences(test_sequences, maxlen=MAXLEN, padding="pre", truncating="pre")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

## 1.7 Why this is sequential data
A bag-of-words representation mainly records which words occur and how often; it does not naturally preserve their order. Compare `the movie was not good` with `the movie was good, not bad`. Negation and word position change meaning. A sequential model reads tokens in order and can build a representation that depends on earlier words.

# Task 2 — RNN intuition, ANN baseline and forward propagation

## 2.1 Order-blind ANN baseline
The embedding converts each token ID into a learned vector. `Flatten` concatenates all positions into one long vector, so Dense layers see fixed position slots rather than a recurrent memory. The model does not reuse the same recurrent transition at each time step. Its first Dense layer's input size grows with `MAXLEN`.

In [ ]:
def build_ann():
    model = models.Sequential([
        layers.Input(shape=(MAXLEN,)),
        layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

ann_model = build_ann()
ann_model.summary()
ann_start = time.time()
ann_history = ann_model.fit(
    X_train, y_train, epochs=5, batch_size=BATCH_SIZE,
    validation_split=0.1, verbose=1
)
ann_seconds = (time.time() - ann_start) / len(ann_history.epoch)
ann_prob = ann_model.predict(X_test, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype(int)

## 2.2 RNN intuition and equations
An RNN reads one time step at a time. At step `t`, it receives the current input vector `x(t)` and previous hidden state `h(t-1)`. The hidden state is a running summary of information read so far.

\[
h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1}+b)
\]

- `x(t)`: current word embedding.
- `h(t-1)`: previous hidden state (past context).
- `Wxh`: weights from input to hidden state.
- `Whh`: recurrent weights from previous hidden state.
- `b`: bias.
- `tanh`: non-linear activation, typically outputs values between -1 and 1.

An output can be computed as:

\[
y_t=f(W_{hy}h_t+b_y)
\]

The same `Wxh`, `Whh` and bias are reused at every time step. This is weight sharing across time. The model therefore has a fixed parameter count for a given input and hidden size, regardless of sequence length.

## 2.3 Manual RNN forward propagation using NumPy
Use input size 2, hidden size 3 and four time steps. The initial hidden state is all zeros. The calculation below explicitly performs the recurrence.

In [ ]:
np.random.seed(42)
input_size, hidden_size, steps = 2, 3, 4
Wxh = np.random.randn(hidden_size, input_size) * 0.5
Whh = np.random.randn(hidden_size, hidden_size) * 0.5
b = np.random.randn(hidden_size) * 0.5
sequence = np.random.randn(steps, input_size) * 0.5

h = np.zeros(hidden_size)
manual_states = []
for t in range(steps):
    h = np.tanh(Wxh @ sequence[t] + Whh @ h + b)
    manual_states.append(h.copy())
    print(f"t={t+1}: {h}")

manual_states = np.array(manual_states)
plt.figure(figsize=(6, 3))
sns.heatmap(manual_states.T, annot=True, cmap="coolwarm", center=0,
            xticklabels=[1,2,3,4], yticklabels=["h1","h2","h3"])
plt.title("Manual RNN Hidden States")
plt.xlabel("Time step"); plt.ylabel("Hidden unit")
plt.tight_layout(); plt.savefig("plots/rnn_forward_heatmap.png", dpi=150); plt.show()

## 2.4 Verify NumPy against Keras
Keras stores its input kernel as `(input_size, units)`, while the NumPy equation above uses `(units, input_size)`. Therefore the input matrix must be transposed. The recurrent matrix is also transposed to match Keras' storage convention.

In [ ]:
keras_rnn_check = layers.SimpleRNN(3, return_sequences=True, use_bias=True)
_ = keras_rnn_check(np.zeros((1, steps, input_size), dtype=np.float32))
keras_rnn_check.set_weights([Wxh.T.astype("float32"), Whh.T.astype("float32"), b.astype("float32")])
keras_output = keras_rnn_check(sequence[np.newaxis, ...].astype("float32")).numpy()[0]
print("NumPy and Keras all close:", np.allclose(manual_states, keras_output, atol=1e-6))
print("Maximum absolute difference:", np.max(np.abs(manual_states - keras_output)))

## 2.5 SimpleRNN parameter count
For a SimpleRNN with `U` units and input size `D`:

\[
Parameters = U(D+U+1)
\]

For 64 units and embedding size 32:

\[
64(32+64+1)=64(97)=6,208
\]

- `U × D`: input-to-hidden weights.
- `U × U`: hidden-to-hidden recurrent weights.
- `U`: bias terms.

This recurrent parameter count does not depend on `MAXLEN`.

In [ ]:
print("Expected SimpleRNN parameters:", 64 * (EMBEDDING_DIM + 64 + 1))

## 2.6 ANN vs RNN
| Feature | ANN baseline | RNN |
|---|---|---|
| Input handling | Fixed vector / flattened positions | One time step at a time |
| Word order | Not naturally modelled as recurrent context | Uses sequence order |
| Weight sharing | Dense weights are position-specific | Same recurrent weights at each step |
| Parameters vs length | Flatten-to-Dense parameters grow with MAXLEN | Recurrent cell parameters do not grow with time steps |
| Earlier words | No recurrent hidden memory | Summarised in hidden state |
| Typical uses | Tabular / fixed-size features | Text, speech, time series |

# Task 3 — Types of RNN and SimpleRNN sentiment model

## 3.1 Four RNN input/output patterns
For a batch of 4 sequences, 10 time steps and 8 features:
- One-to-one: one vector produces one output.
- One-to-many: one vector is repeated/used to generate a sequence.
- Many-to-one: a sequence produces one output vector.
- Many-to-many: a sequence produces an output at every time step.

In [ ]:
# One-to-one
one_to_one = models.Sequential([layers.Input((8,)), layers.Dense(1)])
print("One-to-one:", one_to_one(np.zeros((4,8))).shape)

# One-to-many demonstration: vector -> repeated sequence -> RNN sequence output
one_to_many = models.Sequential([
    layers.Input((8,)), layers.RepeatVector(10),
    layers.SimpleRNN(16, return_sequences=True)
])
print("One-to-many:", one_to_many(np.zeros((4,8))).shape)

# Many-to-one
many_to_one = layers.SimpleRNN(16, return_sequences=False)
print("Many-to-one:", many_to_one(np.zeros((4,10,8))).shape)

# Many-to-many
many_to_many = layers.SimpleRNN(16, return_sequences=True)
print("Many-to-many:", many_to_many(np.zeros((4,10,8))).shape)

| Type | Input shape | Output shape | Sequence output? | Example |
|---|---|---|---|---|
| One-to-one | `(batch, features)` | `(batch, outputs)` | No | Image class prediction |
| One-to-many | `(batch, features)` | `(batch, time, units)` | Yes | Image caption generation |
| Many-to-one | `(batch, time, features)` | `(batch, units)` | No | Sentiment analysis |
| Many-to-many | `(batch, time, features)` | `(batch, time, units)` | Yes | Token tagging / sequence mapping |

This project uses **many-to-one**: the entire movie review is the input sequence and one sentiment label is predicted for the whole review.

## 3.3 Build the SimpleRNN model
Architecture: `Embedding(10000, 32) → SimpleRNN(64) → Dense(1, sigmoid)`. The sigmoid output is a probability for the positive class.

In [ ]:
def build_model(cell_type="rnn", units=64, maxlen=MAXLEN, learning_rate=0.001):
    cell_type = cell_type.lower()
    cell_layer = {
        "rnn": layers.SimpleRNN,
        "lstm": layers.LSTM,
        "gru": layers.GRU
    }[cell_type]
    model = models.Sequential([
        layers.Input(shape=(maxlen,)),
        layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM),
        cell_layer(units),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def train_model(model, Xtr=X_train, ytr=y_train, epochs=EPOCHS):
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )
    start = time.time()
    history = model.fit(
        Xtr, ytr, epochs=epochs, batch_size=BATCH_SIZE,
        validation_split=0.1, callbacks=[early_stop], verbose=1
    )
    seconds_per_epoch = (time.time() - start) / max(1, len(history.epoch))
    return history, seconds_per_epoch

def evaluate_model(model, Xte=X_test, yte=y_test):
    prob = model.predict(Xte, verbose=0).ravel()
    pred = (prob >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(yte, pred),
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
        "f1": f1_score(yte, pred, zero_division=0),
        "roc_auc": roc_auc_score(yte, prob),
        "prob": prob, "pred": pred,
        "report": classification_report(yte, pred, zero_division=0),
        "cm": confusion_matrix(yte, pred)
    }

results = {}
model_objects = {}
histories = {}

# Save ANN baseline results
ann_eval = evaluate_model(ann_model)
results["ANN baseline"] = {
    **{k: ann_eval[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "parameters": ann_model.count_params(),
    "time_per_epoch": ann_seconds,
    "epochs": len(ann_history.epoch)
}

## 3.4 Train SimpleRNN and plot learning curves
EarlyStopping monitors validation loss and restores the best weights. Loss measures prediction error; accuracy measures the fraction of correct labels. Record actual curves and runtime rather than assuming a result.

In [ ]:
rnn_model = build_model("rnn", units=64)
rnn_model.summary()
rnn_history, rnn_time = train_model(rnn_model)
model_objects["SimpleRNN"] = rnn_model
histories["SimpleRNN"] = rnn_history

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(rnn_history.history["loss"], label="train")
ax[0].plot(rnn_history.history["val_loss"], label="validation")
ax[0].set_title("SimpleRNN Loss"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend()
ax[1].plot(rnn_history.history["accuracy"], label="train")
ax[1].plot(rnn_history.history["val_accuracy"], label="validation")
ax[1].set_title("SimpleRNN Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

rnn_eval = evaluate_model(rnn_model)
print("Test accuracy:", rnn_eval["accuracy"])
print(rnn_eval["report"])
sns.heatmap(rnn_eval["cm"], annot=True, fmt="d", cmap="Blues")
plt.title("SimpleRNN Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()
results["SimpleRNN"] = {
    **{k: rnn_eval[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "parameters": rnn_model.count_params(),
    "time_per_epoch": rnn_time,
    "epochs": len(rnn_history.epoch)
}

## 3.5 ROC-AUC
The ROC curve plots true-positive rate against false-positive rate over different classification thresholds. ROC-AUC measures ranking/separation across thresholds; unlike accuracy at one threshold, it is not tied to only the 0.5 cutoff.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, rnn_eval["prob"])
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"SimpleRNN AUC={rnn_eval['roc_auc']:.3f}")
plt.plot([0,1], [0,1], "--", label="Random")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("SimpleRNN ROC Curve"); plt.legend(); plt.grid(True); plt.show()

## 3.6 Embedding layer
An embedding matrix maps each integer token ID to a trainable dense vector. The model learns these vectors from the training objective; this practical does not use pretrained GloVe, Word2Vec or BERT vectors. With vocabulary size 10,000 and embedding dimension 32, the embedding matrix has shape `(10000, 32)`.

In [ ]:
embedding_weights = rnn_model.layers[0].get_weights()[0]
print("Embedding matrix shape:", embedding_weights.shape)
for word in ["good", "bad"]:
    idx = tokenizer.word_index.get(word)
    if idx is not None and idx < VOCAB_SIZE:
        print(word, "index:", idx, "vector:", embedding_weights[idx][:10])

# Task 4 — Backpropagation Through Time and RNN problems

## 4.1 BPTT
Backpropagation Through Time (BPTT) trains an RNN by unrolling it across its time steps. The unrolled network looks like a deep network whose repeated cells share the same weights. For a final-step loss, gradients travel backwards through the recurrent chain.

The gradient reaching an early hidden state contains a product of Jacobian terms. Conceptually:

\[
\frac{\partial L}{\partial h_1}
=
\frac{\partial L}{\partial h_T}
\prod_{t=2}^{T}
\left(W_{hh}^{T}\,diag(1-h_t^2)\right)
\]

The gradient with respect to a shared recurrent weight is the sum of its contributions from each time step. Long products can become extremely small or large.

## 4.2 Vanishing and exploding gradients
- **Vanishing:** if repeated factors have magnitude below 1, their product shrinks exponentially. Early words receive tiny gradients and have little influence on learning.
- **Exploding:** if repeated factors have magnitude above 1, the product can grow exponentially. Training may show unstable loss spikes or NaN values.
- **Practical impact:** a basic SimpleRNN often struggles to preserve useful information over long distances. The assignment describes its effective memory as roughly 10–20 time steps; treat this as a practical rule of thumb, not a universal hard limit.

## 4.3 Gradient flow with GradientTape
This diagnostic explicitly exposes the sequence of hidden states and differentiates the final loss with respect to them. The plotted norm is averaged over the batch at each time step. A rapidly smaller norm at early steps is evidence of weak gradient flow.

In [ ]:
def gradient_norms_rnn_family(cell_type, model, Xbatch, ybatch):
    # Reuse trained embedding, recurrent cell and output layer weights.
    embedding_layer = model.layers[0]
    recurrent_layer = model.layers[1]
    output_layer = model.layers[-1]

    if cell_type == "rnn":
        sequence_layer = layers.SimpleRNN(recurrent_layer.units, return_sequences=True)
    elif cell_type == "lstm":
        sequence_layer = layers.LSTM(recurrent_layer.units, return_sequences=True)
    else:
        sequence_layer = layers.GRU(recurrent_layer.units, return_sequences=True)

    xb = tf.convert_to_tensor(Xbatch, dtype=tf.int32)
    yb = tf.convert_to_tensor(ybatch.reshape(-1, 1), dtype=tf.float32)

    with tf.GradientTape() as tape:
        embedded = embedding_layer(xb)
        H = sequence_layer(embedded)
        sequence_layer.set_weights(recurrent_layer.get_weights())
        # Recompute with the trained recurrent weights before taking the gradient.
        H = sequence_layer(embedded)
        tape.watch(H)
        last_state = H[:, -1, :]
        prediction = output_layer(last_state)
        loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(yb, prediction))

    grads = tape.gradient(loss, H)
    norms = tf.reduce_mean(tf.norm(grads, axis=-1), axis=0).numpy()
    return norms

## 4.4 Accuracy vs sequence length
Re-pad the same token sequences at lengths 50, 100, 200 and 300. For a fair comparison, use the same split and seed. This experiment costs time because it trains four separate models.

In [ ]:
sequence_length_results = {}
for length in [50, 100, 200, 300]:
    Xtr_len = pad_sequences(train_sequences, maxlen=length, padding="pre", truncating="pre")
    Xte_len = pad_sequences(test_sequences, maxlen=length, padding="pre", truncating="pre")
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    m = build_model("rnn", units=64, maxlen=length)
    h, sec = train_model(m, Xtr_len, y_train, epochs=8)
    ev = evaluate_model(m, Xte_len, y_test)
    sequence_length_results[length] = ev["accuracy"]
    print(length, ev["accuracy"])
plt.figure(figsize=(6,4))
plt.plot(list(sequence_length_results.keys()), list(sequence_length_results.values()), marker="o")
plt.xlabel("Sequence length"); plt.ylabel("Test accuracy")
plt.title("SimpleRNN Accuracy vs Sequence Length"); plt.grid(True); plt.show()

## 4.5 Exploding-gradient demonstration and clipping
A deliberately high learning rate can destabilise training. Gradient clipping limits the gradient norm; it helps control exploding gradients but does not restore information lost through vanishing gradients.

In [ ]:
def build_rnn_with_optimizer(learning_rate, clipnorm=None):
    opt = tf.keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=clipnorm)
    m = models.Sequential([
        layers.Input((MAXLEN,)),
        layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM),
        layers.SimpleRNN(64),
        layers.Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    return m

unstable_model = build_rnn_with_optimizer(0.05)
clipped_model = build_rnn_with_optimizer(0.05, clipnorm=1.0)
unstable_hist = unstable_model.fit(X_train, y_train, epochs=5, batch_size=BATCH_SIZE,
                                   validation_split=0.1, verbose=1)
clipped_hist = clipped_model.fit(X_train, y_train, epochs=5, batch_size=BATCH_SIZE,
                                 validation_split=0.1, verbose=1)
plt.figure(figsize=(7,4))
plt.plot(unstable_hist.history["loss"], marker="o", label="High LR, no clipping")
plt.plot(clipped_hist.history["loss"], marker="o", label="High LR, clipnorm=1.0")
plt.xlabel("Epoch"); plt.ylabel("Training loss")
plt.title("Effect of Gradient Clipping"); plt.legend(); plt.show()

## 4.6 Problems summary
| Problem | Cause | Possible symptom | Remedy |
|---|---|---|---|
| Vanishing gradient | Repeated small derivatives/Jacobians | Early tokens have tiny gradient norms | LSTM / GRU gates |
| Exploding gradient | Repeated large factors | Loss spikes or NaN | Gradient clipping |
| Slow sequential training | Time steps are processed sequentially | Longer training time | Shorter sequences / GPU |
| Short-term memory | Basic hidden state struggles with long dependencies | Long-distance context is missed | Gated recurrent cells |

# Task 5 — LSTM (Long Short-Term Memory)

## 5.1 LSTM cell theory
LSTM adds a cell state `C(t)` and gates to control information flow. Think of the cell state as a conveyor belt that can carry information across many time steps.

\[
f_t=\sigma(W_f[h_{t-1},x_t]+b_f)
\]
\[
i_t=\sigma(W_i[h_{t-1},x_t]+b_i),\quad
g_t=\tanh(W_g[h_{t-1},x_t]+b_g)
\]
\[
C_t=f_t*C_{t-1}+i_t*g_t
\]
\[
o_t=\sigma(W_o[h_{t-1},x_t]+b_o),\quad
h_t=o_t*\tanh(C_t)
\]

- **Forget gate:** decides what old information to keep or discard.
- **Input gate:** decides how much new information to write.
- **Candidate state:** proposes new content.
- **Output gate:** decides what part of the cell state becomes the visible hidden state.

## 5.2 Why LSTM helps with vanishing gradients
The cell state has an additive update. Along this path, the gradient is multiplied by the forget gate rather than repeatedly passing only through a recurrent weight matrix and a tanh derivative. When the forget gate is near 1, information and gradients can travel across more steps. This mitigates vanishing gradients; it does not guarantee perfect long-term memory.

## 5.3 LSTM parameter count
An LSTM has four parameter sets: forget, input, candidate and output.

\[
Parameters=4U(D+U+1)
\]

For 64 units and 32-dimensional embeddings:

\[
4\times64\times(32+64+1)=24,832
\]

This is four times the SimpleRNN cell parameter count for the same dimensions.

In [ ]:
print("Expected LSTM parameters:", 4 * 64 * (EMBEDDING_DIM + 64 + 1))

## 5.4 Manual single-step LSTM using NumPy
The following small implementation demonstrates the gate calculations over four steps. It is an educational cell calculation, separate from Keras training.

In [ ]:
np.random.seed(42)
D, U, T = 2, 3, 4
def rand_gate():
    return np.random.randn(U, D+U) * 0.3, np.random.randn(U) * 0.3
Wf,bf = rand_gate(); Wi,bi = rand_gate()
Wg,bg = rand_gate(); Wo,bo = rand_gate()
xs = np.random.randn(T,D) * 0.5
h = np.zeros(U); C = np.zeros(U)
manual_lstm_rows = []
sigmoid = lambda z: 1/(1+np.exp(-z))
for t, x in enumerate(xs, 1):
    joined = np.concatenate([h,x])
    f = sigmoid(Wf @ joined + bf)
    i = sigmoid(Wi @ joined + bi)
    g = np.tanh(Wg @ joined + bg)
    o = sigmoid(Wo @ joined + bo)
    C = f*C + i*g
    h = o*np.tanh(C)
    manual_lstm_rows.append({"step":t, "f":f.copy(), "i":i.copy(),
                             "o":o.copy(), "C":C.copy(), "h":h.copy()})
    print(f"Step {t}: f={f.round(3)}, i={i.round(3)}, o={o.round(3)}, C={C.round(3)}, h={h.round(3)}")

## 5.5 Build, train and evaluate LSTM
Use the same embedding size, hidden units, optimiser, batch size, validation split and early stopping settings as SimpleRNN. This makes the comparison more meaningful.

In [ ]:
lstm_model = build_model("lstm", units=64)
lstm_model.summary()
lstm_history, lstm_time = train_model(lstm_model)
model_objects["LSTM"] = lstm_model
histories["LSTM"] = lstm_history
lstm_eval = evaluate_model(lstm_model)
print(lstm_eval["report"])
sns.heatmap(lstm_eval["cm"], annot=True, fmt="d", cmap="Blues")
plt.title("LSTM Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()
results["LSTM"] = {
    **{k: lstm_eval[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "parameters": lstm_model.count_params(),
    "time_per_epoch": lstm_time,
    "epochs": len(lstm_history.epoch)
}

## 5.6 Compare gradient flow: SimpleRNN vs LSTM
The diagnostic plot compares gradients at each sequence position. Use the measured earliest-step values in your report; do not claim a numerical improvement until your run produces it.

In [ ]:
lstm_grad = gradient_norms_rnn_family("lstm", lstm_model, X_test[:sample_n], y_test[:sample_n])
plt.figure(figsize=(8,4))
plt.plot(rnn_grad, label="SimpleRNN")
plt.plot(lstm_grad, label="LSTM")
plt.yscale("log"); plt.xlabel("Time step index"); plt.ylabel("Mean gradient norm (log scale)")
plt.title("Gradient Flow: SimpleRNN vs LSTM"); plt.legend(); plt.grid(True); plt.show()
print("Earliest-step gradient norms — RNN:", rnn_grad[0], "LSTM:", lstm_grad[0])

## 5.7 Stacked LSTM
The first LSTM must use `return_sequences=True` because the second LSTM needs a full sequence of hidden states, not only one final vector. The second layer returns one vector for the final sentiment classifier. Extra depth increases parameters and training time and may or may not improve generalisation.

In [ ]:
stacked_lstm = models.Sequential([
    layers.Input((MAXLEN,)),
    layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid")
])
stacked_lstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
stacked_history, stacked_time = train_model(stacked_lstm)
stacked_eval = evaluate_model(stacked_lstm)
model_objects["Stacked LSTM"] = stacked_lstm
histories["Stacked LSTM"] = stacked_history
results["Stacked LSTM"] = {
    **{k: stacked_eval[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "parameters": stacked_lstm.count_params(),
    "time_per_epoch": stacked_time,
    "epochs": len(stacked_history.epoch)
}
print(stacked_eval["report"])

# Task 6 — GRU (Gated Recurrent Unit)

## 6.1 GRU theory
A GRU is a gated recurrent cell with two gates and no separate cell state.

\[
z_t=\sigma(W_z[h_{t-1},x_t])
\]
\[
r_t=\sigma(W_r[h_{t-1},x_t])
\]
\[
\tilde h_t=\tanh(W[r_t*h_{t-1},x_t])
\]
\[
h_t=(1-z_t)*h_{t-1}+z_t*\tilde h_t
\]

- **Update gate:** controls how much old state is retained versus replaced by the candidate.
- **Reset gate:** controls how much past information is used to form the candidate.
- GRU combines the roles of the LSTM forget and input gates into an update gate and does not maintain a separate cell state.

## 6.2 GRU vs LSTM
| Feature | GRU | LSTM |
|---|---|---|
| Main gates | Update and reset | Forget, input and output (plus candidate) |
| Separate cell state | No | Yes |
| Parameter sets | 3 | 4 |
| Parameter count | Usually fewer | Usually more |
| Training speed | Often faster | Can be slower |
| When useful | When a simpler gated model is preferred | When separate long-term cell memory is useful |

Actual speed and quality depend on hardware, implementation, data and hyperparameters.

## 6.3 GRU parameter count (Keras reset_after=True)
With two bias vectors per gate set:

\[
Parameters=3(UD+U^2+2U)
\]

For 64 units and input size 32:

\[
3(64\times32+64\times64+2\times64)=18,816
\]

This is approximately three quarters of the LSTM cell parameter count.

In [ ]:
print("Expected GRU parameters:", 3 * (64*EMBEDDING_DIM + 64*64 + 2*64))

## 6.4 Build, train and evaluate GRU

In [ ]:
gru_model = build_model("gru", units=64)
gru_model.summary()
gru_history, gru_time = train_model(gru_model)
model_objects["GRU"] = gru_model
histories["GRU"] = gru_history
gru_eval = evaluate_model(gru_model)
print(gru_eval["report"])
sns.heatmap(gru_eval["cm"], annot=True, fmt="d", cmap="Blues")
plt.title("GRU Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()
results["GRU"] = {
    **{k: gru_eval[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "parameters": gru_model.count_params(),
    "time_per_epoch": gru_time,
    "epochs": len(gru_history.epoch)
}

## 6.5 Gradient flow across all three recurrent cells

In [ ]:
gru_grad = gradient_norms_rnn_family("gru", gru_model, X_test[:sample_n], y_test[:sample_n])
plt.figure(figsize=(9,5))
plt.plot(rnn_grad, label="SimpleRNN")
plt.plot(lstm_grad, label="LSTM")
plt.plot(gru_grad, label="GRU")
plt.yscale("log"); plt.xlabel("Time step index"); plt.ylabel("Mean gradient norm (log scale)")
plt.title("Gradient Flow Comparison: SimpleRNN, LSTM and GRU")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig("plots/gradient_norms_all_cells.png", dpi=150); plt.show()

## 6.6 Hidden-unit sensitivity
Train LSTM and GRU at 32, 64 and 128 units. More units increase representational capacity and parameter count, but returns may diminish and overfitting or slower training may occur.

In [ ]:
sensitivity_rows = []
for cell in ["lstm", "gru"]:
    for units in [32, 64, 128]:
        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED)
        m = build_model(cell, units=units)
        h, sec = train_model(m, epochs=6)
        ev = evaluate_model(m)
        sensitivity_rows.append({
            "Cell": cell.upper(), "Units": units,
            "Accuracy": ev["accuracy"], "Parameters": m.count_params(),
            "Time per epoch": sec
        })
sensitivity_df = pd.DataFrame(sensitivity_rows)
display(sensitivity_df)
fig, ax = plt.subplots(1,2,figsize=(11,4))
sns.lineplot(data=sensitivity_df, x="Units", y="Accuracy", hue="Cell", marker="o", ax=ax[0])
ax[0].set_title("Test Accuracy vs Hidden Units")
sns.lineplot(data=sensitivity_df, x="Units", y="Parameters", hue="Cell", marker="o", ax=ax[1])
ax[1].set_title("Parameter Count vs Hidden Units")
plt.tight_layout(); plt.show()

# Task 7 — Final comparison, inference and recommendation

## 7.1 Full results table
The table uses actual metrics measured by this notebook. Precision, recall and F1 are calculated at threshold 0.5; ROC-AUC uses probabilities.

In [ ]:
# Add recurrent model predictions and ensure all model results are available
eval_by_name = {
    "ANN baseline": ann_eval, "SimpleRNN": rnn_eval,
    "LSTM": lstm_eval, "Stacked LSTM": stacked_eval, "GRU": gru_eval
}
rows = []
for name, r in results.items():
    rows.append({
        "Model": name, "Parameters": r["parameters"],
        "Time per Epoch (s)": r["time_per_epoch"], "Epochs Trained": r["epochs"],
        "Test Accuracy": r["accuracy"], "Precision": r["precision"],
        "Recall": r["recall"], "F1": r["f1"], "ROC-AUC": r["roc_auc"]
    })
results_df = pd.DataFrame(rows)
display(results_df.style.highlight_max(
    subset=["Test Accuracy","Precision","Recall","F1","ROC-AUC"], color="#ffcccc"
))
results_df.to_csv("plots/results_table.csv", index=False)

## 7.2 Combined ROC curves
ROC curves compare threshold behaviour for ANN, SimpleRNN, LSTM and GRU. The stacked model is included in the metrics table; the assignment requests the four named ROC curves below.

In [ ]:
plt.figure(figsize=(7,6))
for name in ["ANN baseline", "SimpleRNN", "LSTM", "GRU"]:
    ev = eval_by_name[name]
    fpr, tpr, _ = roc_curve(y_test, ev["prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={ev['roc_auc']:.3f})")
plt.plot([0,1],[0,1],"k--",label="Random")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison"); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.savefig("plots/roc_curves.png", dpi=150); plt.show()

## 7.3 Robustness by review length
Create short, medium and long groups using the assignment boundaries: short below 100 words, medium 100–250 words, long above 250 words. Evaluate the best gated model and SimpleRNN on the same selected test rows.

In [ ]:
test_lengths = np.array([len(str(t).split()) for t in X_test_text])
groups = {
    "Short (<100)": test_lengths < 100,
    "Medium (100–250)": (test_lengths >= 100) & (test_lengths <= 250),
    "Long (>250)": test_lengths > 250
}
best_gated_name = max(["LSTM","GRU"], key=lambda n: results[n]["f1"])
best_gated_model = model_objects[best_gated_name]
robust_rows = []
for group_name, mask in groups.items():
    if mask.sum() == 0:
        continue
    for model_name, model in [("SimpleRNN", rnn_model), (best_gated_name, best_gated_model)]:
        # Both use the same MAXLEN in this main comparison.
        prob = model.predict(X_test[mask], verbose=0).ravel()
        pred = (prob >= 0.5).astype(int)
        robust_rows.append({
            "Length group": group_name, "Model": model_name,
            "Accuracy": accuracy_score(y_test[mask], pred), "Count": int(mask.sum())
        })
robust_df = pd.DataFrame(robust_rows)
display(robust_df)
sns.barplot(data=robust_df, x="Length group", y="Accuracy", hue="Model")
plt.title("Accuracy by Review Length"); plt.xlabel("Review length group")
plt.ylabel("Accuracy"); plt.tight_layout()
plt.savefig("plots/accuracy_by_length.png", dpi=150); plt.show()

## 7.4 Error analysis: 4 false positives and 4 false negatives
A false positive is a negative review predicted positive. A false negative is a positive review predicted negative. Inspect the examples rather than assuming the model's mistake has one cause.

In [ ]:
best_eval = eval_by_name[best_gated_name]
best_pred = best_eval["pred"]
best_prob = best_eval["prob"]
error_df = pd.DataFrame({
    "text": X_test_text, "true": y_test, "pred": best_pred, "prob_positive": best_prob
})
fp = error_df[(error_df.true == 0) & (error_df.pred == 1)].head(4)
fn = error_df[(error_df.true == 1) & (error_df.pred == 0)].head(4)
print("FALSE POSITIVES")
for _, row in fp.iterrows():
    print("\\nTrue: negative | Predicted probability positive:", round(row.prob_positive,4))
    print(row.text[:1200])
print("\\nFALSE NEGATIVES")
for _, row in fn.iterrows():
    print("\\nTrue: positive | Predicted probability positive:", round(row.prob_positive,4))
    print(row.text[:1200])

**Patterns to look for:** sarcasm, negation, mixed opinions, very long reviews, comparisons, and reviews describing a bad character or event in an otherwise good film. These are analysis prompts; only report patterns actually visible in your selected errors.

## 7.5 Inference function
The function must reuse the same cleaning, tokenizer and padding settings used during training. The returned probability is the model's estimated probability of the positive class.

In [ ]:
def predict_sentiment(text, model=None):
    if model is None:
        model = best_gated_model
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAXLEN, padding="pre", truncating="pre")
    probability = float(model.predict(padded, verbose=0)[0][0])
    label = "positive" if probability >= 0.5 else "negative"
    return label, probability

custom_sentences = [
    "This film was wonderful and I loved every moment.",
    "The movie was boring and badly acted.",
    "This film is not good.",
    "I do not think it was not enjoyable.",
    "Brilliant movie, if your idea of fun is watching paint dry.",
    "Amazing!"
]
inference_rows = []
for sentence in custom_sentences:
    label, prob = predict_sentiment(sentence)
    inference_rows.append({"Review":sentence, "Predicted sentiment":label,
                           "Positive probability":round(prob,4)})
display(pd.DataFrame(inference_rows))

## 7.6 Which recurrent cell should we use?
Write the final recommendation only after examining your actual results table and plots.

A defensible recommendation should discuss:
1. Accuracy, F1 and ROC-AUC ranking — use measured values.
2. Training time and parameter trade-off — LSTM generally has more parameters than GRU; measured runtime depends on hardware.
3. Long-review behaviour — compare the length-group chart and gradient plots.
4. Production choice — choose the model that best balances predictive quality, latency, memory and error costs for the intended service.
5. Limitations — embeddings are learned only from this dataset (no pretrained knowledge); the models read in one direction and may struggle with sarcasm, ambiguity and very long context.

**Template:** “In my experiment, ______ achieved the best ______. ______ required ______ parameters and ______ seconds per epoch. Based on the measured quality and computational cost, I recommend ______ for this sentiment task because ______. The main limitations are ______ and ______.”

## 7.7 Final notebook quality and export
Run **Kernel → Restart & Run All** and resolve every error. Confirm all figures and outputs exist. Export the notebook to HTML after successful execution.

In [ ]:
# Environment / package list for requirements.txt
requirements = [
    "tensorflow",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn"
]
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("\\n".join(requirements) + "\\n")
print("requirements.txt created.")
print("Plots folder:", os.listdir("plots"))

# Project conclusion (complete after running)
The project compared an order-blind ANN with SimpleRNN, LSTM, Stacked LSTM and GRU for binary IMDB review sentiment classification. The data was cleaned, split before tokenizer fitting, converted to integer sequences and padded. The experiments examined recurrent forward propagation, BPTT, gradient flow, gated memory, model complexity and inference. Insert your measured best model, metrics and practical recommendation here.

# References
- Maas, A. L. et al. (2011). *Learning Word Vectors for Sentiment Analysis*. The original Large Movie Review Dataset.
- Kaggle: IMDB Dataset of 50K Movie Reviews — https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
- TensorFlow / Keras documentation: https://www.tensorflow.org/